# MAMBA-GINR for Semantic Segmentation on PASCAL VOC 2012

## Experiment Design

**Research Question**: Do learned modulation features from reconstruction pretraining improve semantic segmentation?

**Protocol**:
1. **Pretraining**: Train MAMBA-GINR on 64×64 image reconstruction (no segmentation labels)
2. **Feature Extraction**: Extract per-pixel modulation features (64×64×256)
3. **Segmentation Baseline**: Train U-Net on raw RGB (64×64×3) 
4. **Segmentation Proposed**: Train U-Net on modulation features (64×64×256)
5. **Compare**: IoU, pixel accuracy, class-wise performance

**Hypothesis**: If reconstruction learns semantically meaningful features, modulation-based segmentation should outperform RGB baseline!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import einops
import math
from PIL import Image
import os
from pathlib import Path

from mamba_ssm import Mamba
from mamba_ssm.modules.block import Block

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. PASCAL VOC 2012 Dataset

## 1. Download PASCAL VOC 2012 Dataset

In [ ]:
"""
PASCAL VOC 2012 Dataset Download

This cell downloads the PASCAL VOC 2012 Segmentation dataset.
Dataset size: ~2GB
Download time: 5-10 minutes depending on internet speed

The dataset will be downloaded to: ./data/VOCdevkit/VOC2012/
"""

import os
from pathlib import Path

# Configuration
DATA_ROOT = './data'
VOC_ROOT = Path(DATA_ROOT) / 'VOCdevkit' / 'VOC2012'

print("="*80)
print("PASCAL VOC 2012 Dataset Download")
print("="*80)

# Check if dataset already exists
if VOC_ROOT.exists():
    # Verify key directories
    required_dirs = ['JPEGImages', 'SegmentationClass', 'ImageSets/Segmentation']
    all_exist = all((VOC_ROOT / d).exists() for d in required_dirs)
    
    if all_exist:
        print("\n✓ Dataset already exists at:", VOC_ROOT)
        print("\nDataset structure:")
        print(f"  - Images: {VOC_ROOT / 'JPEGImages'}")
        print(f"  - Segmentation masks: {VOC_ROOT / 'SegmentationClass'}")
        print(f"  - Train/val splits: {VOC_ROOT / 'ImageSets' / 'Segmentation'}")
        
        # Count files
        num_images = len(list((VOC_ROOT / 'JPEGImages').glob('*.jpg')))
        num_masks = len(list((VOC_ROOT / 'SegmentationClass').glob('*.png')))
        print(f"\n  Total images: {num_images}")
        print(f"  Total masks: {num_masks}")
    else:
        print("\n⚠️  Dataset directory exists but is incomplete. Re-downloading...")
        import shutil
        shutil.rmtree(VOC_ROOT.parent, ignore_errors=True)
        all_exist = False
else:
    all_exist = False

# Download if needed
if not all_exist:
    print("\n📥 Downloading PASCAL VOC 2012 dataset...")
    print("   This may take 5-10 minutes (~2GB download)")
    print("   Dataset will be saved to:", DATA_ROOT)
    
    try:
        # Download using torchvision
        from torchvision.datasets import VOCSegmentation
        
        # Download train split (this downloads the full dataset)
        print("\n   Downloading train split...")
        train_dataset = VOCSegmentation(
            root=DATA_ROOT,
            year='2012',
            image_set='train',
            download=True
        )
        
        # Download val split metadata (dataset already downloaded)
        print("   Loading validation split...")
        val_dataset = VOCSegmentation(
            root=DATA_ROOT,
            year='2012',
            image_set='val',
            download=False
        )
        
        print("\n✓ Download complete!")
        
        # Verify download
        print("\nVerifying download...")
        required_dirs = ['JPEGImages', 'SegmentationClass', 'ImageSets/Segmentation']
        verification_passed = all((VOC_ROOT / d).exists() for d in required_dirs)
        
        if verification_passed:
            num_images = len(list((VOC_ROOT / 'JPEGImages').glob('*.jpg')))
            num_masks = len(list((VOC_ROOT / 'SegmentationClass').glob('*.png')))
            
            print("✓ Verification passed!")
            print(f"\nDataset statistics:")
            print(f"  - Train samples: {len(train_dataset)}")
            print(f"  - Val samples: {len(val_dataset)}")
            print(f"  - Total images: {num_images}")
            print(f"  - Total masks: {num_masks}")
        else:
            print("❌ Verification failed! Some directories are missing.")
            print("   Please check your internet connection and try again.")
            
    except Exception as e:
        print(f"\n❌ Download failed with error: {e}")
        print("   Please check your internet connection and try again.")
        raise

print("\n" + "="*80)
print("Dataset ready for use!")
print("="*80)


## 2. PASCAL VOC 2012 Dataset Class

In [ ]:
class VOCSegmentationDataset(Dataset):
    """
    PASCAL VOC 2012 Segmentation Dataset
    Resizes images and masks to 64×64
    
    Classes (21 total):
    0: background
    1-20: aeroplane, bicycle, bird, boat, bottle, bus, car, cat, chair, cow,
          diningtable, dog, horse, motorbike, person, pottedplant, sheep, sofa, train, tvmonitor
    255: ignore/boundary
    """
    def __init__(self, root='./data', split='train', image_size=64):
        self.root = Path(root)
        self.split = split
        self.image_size = image_size
        
        # Download dataset if needed
        voc_root = self.root / 'VOCdevkit' / 'VOC2012'
        if not voc_root.exists():
            print("Downloading PASCAL VOC 2012...")
            torchvision.datasets.VOCSegmentation(
                root=str(self.root),
                year='2012',
                image_set=split,
                download=True
            )
        
        # Load image and mask paths
        split_file = voc_root / 'ImageSets' / 'Segmentation' / f'{split}.txt'
        with open(split_file, 'r') as f:
            self.image_ids = [line.strip() for line in f.readlines()]
        
        self.image_dir = voc_root / 'JPEGImages'
        self.mask_dir = voc_root / 'SegmentationClass'
        
        print(f"Loaded {len(self.image_ids)} images for {split} split")
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        
        # Load image
        img_path = self.image_dir / f'{img_id}.jpg'
        image = Image.open(img_path).convert('RGB')
        
        # Load mask
        mask_path = self.mask_dir / f'{img_id}.png'
        mask = Image.open(mask_path)
        
        # Resize to 64×64
        image = image.resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = mask.resize((self.image_size, self.image_size), Image.NEAREST)
        
        # Convert to tensors
        image = transforms.ToTensor()(image)  # (3, 64, 64)
        mask = torch.from_numpy(np.array(mask)).long()  # (64, 64)
        
        # Map ignore class (255) to 21
        mask[mask == 255] = 21
        
        return image, mask


# Load datasets
train_dataset = VOCSegmentationDataset(root='./data', split='train', image_size=64)
val_dataset = VOCSegmentationDataset(root='./data', split='val', image_size=64)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# Visualize samples
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
    'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant',
    'sheep', 'sofa', 'train', 'tvmonitor', 'ignore'
]

# Color map for visualization
def get_color_map():
    cmap = np.zeros((22, 3), dtype=np.uint8)
    cmap[0] = [0, 0, 0]  # background
    cmap[1] = [128, 0, 0]  # aeroplane
    cmap[2] = [0, 128, 0]  # bicycle
    cmap[3] = [128, 128, 0]  # bird
    cmap[4] = [0, 0, 128]  # boat
    cmap[5] = [128, 0, 128]  # bottle
    cmap[6] = [0, 128, 128]  # bus
    cmap[7] = [128, 128, 128]  # car
    cmap[8] = [64, 0, 0]  # cat
    cmap[9] = [192, 0, 0]  # chair
    cmap[10] = [64, 128, 0]  # cow
    cmap[11] = [192, 128, 0]  # diningtable
    cmap[12] = [64, 0, 128]  # dog
    cmap[13] = [192, 0, 128]  # horse
    cmap[14] = [64, 128, 128]  # motorbike
    cmap[15] = [192, 128, 128]  # person
    cmap[16] = [0, 64, 0]  # pottedplant
    cmap[17] = [128, 64, 0]  # sheep
    cmap[18] = [0, 192, 0]  # sofa
    cmap[19] = [128, 192, 0]  # train
    cmap[20] = [0, 64, 128]  # tvmonitor
    cmap[21] = [224, 224, 192]  # ignore
    return cmap

color_map = get_color_map()

def mask_to_rgb(mask):
    """Convert class mask to RGB for visualization"""
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls_id in range(22):
        rgb[mask == cls_id] = color_map[cls_id]
    return rgb

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img, mask = train_dataset[i]
    axes[0, i].imshow(img.permute(1, 2, 0))
    axes[0, i].set_title('Image')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(mask_to_rgb(mask.numpy()))
    axes[1, i].set_title('Segmentation Mask')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 3. MAMBA-GINR Architecture (same as CIFAR-10)

In [ ]:
# Copy all architecture components from cifar10_experiments_CORRECTED.ipynb
# BiMamba, MambaEncoder, ImplicitSequentialBias, LAINRDecoder, MambaGINR_CIFAR
# (Same code as before - I'll include the full classes)

# ============================================================================
# BiMamba and Encoder
# ============================================================================

class BiMamba(nn.Module):
    """Bidirectional Mamba processing"""
    def __init__(self, dim=256):
        super().__init__()
        self.f_mamba = Mamba(d_model=dim)
        self.r_mamba = Mamba(d_model=dim)
    
    def forward(self, x, **kwargs):
        x_f = self.f_mamba(x, **kwargs)
        x_r = torch.flip(self.r_mamba(torch.flip(x, dims=[1]), **kwargs), dims=[1])
        return (x_f + x_r) / 2


class MambaEncoder(nn.Module):
    """Stack of Mamba blocks"""
    def __init__(self, depth=6, dim=256, ff_dim=1024, dropout=0.):
        super().__init__()
        self.blocks = nn.ModuleList([
            Block(
                dim=dim,
                mixer_cls=lambda d: BiMamba(d),
                mlp_cls=lambda d: nn.Sequential(
                    nn.Linear(d, ff_dim),
                    nn.GELU(),
                    nn.Dropout(dropout),
                    nn.Linear(ff_dim, d),
                    nn.Dropout(dropout),
                ),
                norm_cls=nn.LayerNorm,
                fused_add_norm=False
            )
            for _ in range(depth)
        ])
    
    def forward(self, x):
        residual = None
        for block in self.blocks:
            x, residual = block(x, residual=residual)
        return x


class ImplicitSequentialBias(nn.Module):
    """Learnable Position Tokens"""
    def __init__(self, num_lp=256, dim=256, input_len=256, type='equidistant'):
        super().__init__()
        self.num_lp = num_lp
        self.dim = dim
        self.type = type
        
        self.lps = nn.Parameter(torch.randn(num_lp, dim) * 0.02)
        self.lp_idxs = self._compute_lp_indices(input_len, num_lp, type)
        self.perm = self._compute_permutation(input_len, num_lp)
    
    def _compute_lp_indices(self, seq_len, num_lp, type):
        total_len = seq_len + num_lp
        if type == 'equidistant':
            return torch.linspace(0, total_len - 1, steps=num_lp).long()
        else:
            return torch.linspace(0, total_len - 1, steps=num_lp).long()
    
    def _compute_permutation(self, seq_len, num_lp):
        total_len = seq_len + num_lp
        perm = torch.full((total_len,), -1, dtype=torch.long)
        perm[self.lp_idxs] = torch.arange(seq_len, seq_len + num_lp)
        perm[perm == -1] = torch.arange(seq_len)
        return perm
    
    def add_lp(self, x):
        B = x.shape[0]
        lps = einops.repeat(self.lps, 'n d -> b n d', b=B)
        x_full = torch.cat([x, lps], dim=1)
        return x_full[:, self.perm]
    
    def extract_lp(self, x):
        return x[:, self.lp_idxs]


print("✓ Encoder components defined")

In [ ]:
# ============================================================================
# LAINR Decoder (continues from previous cell)
# ============================================================================

def exists(val):
    return val is not None

def default(val, d):
    return val if exists(val) else d


class SharedTokenCrossAttention(nn.Module):
    def __init__(self, query_dim, context_dim=None, heads=2, dim_head=64):
        super().__init__()
        context_dim = default(context_dim, query_dim)
        inner_dim = dim_head * heads
        self.heads = heads
        self.dim_head = dim_head
        self.scale = dim_head ** -0.5

        self.to_q = nn.Linear(query_dim, inner_dim, bias=False)
        self.to_kv = nn.Linear(context_dim, inner_dim * 2, bias=False)
        self.to_out = nn.Linear(inner_dim, query_dim)

    def forward(self, x, context, bias=None):
        B, HW, D = x.shape
        H = self.heads
        Dh = self.dim_head
        D_inner = H * Dh

        q = self.to_q(x)
        kv = self.to_kv(context)
        k, v = kv.chunk(2, dim=-1)

        q = q.view(B, HW, H, Dh).transpose(1, 2)
        k = k.view(B, -1, H, Dh).transpose(1, 2)
        v = v.view(B, -1, H, Dh).transpose(1, 2)

        sim = torch.matmul(q, k.transpose(-1, -2)) * self.scale
        
        if bias is not None:
            bias = einops.repeat(bias, 'b l n -> b h l n', h=H)
            bias = bias.transpose(-2, -1)
            sim = sim + bias
        
        attn = sim.softmax(dim=-1)
        out = torch.matmul(attn, v)

        out = out.transpose(1, 2).contiguous().view(B, HW, D_inner)
        out = self.to_out(out)
        return out


class LAINRDecoder(nn.Module):
    def __init__(self, feature_dim=64, input_dim=2, output_dim=3, 
                 sigma_q=16, sigma_ls=[128, 32], n_patches=1024, hidden_dim=256, context_dim=256):
        super().__init__()
        self.layer_num = len(sigma_ls)
        self.n = feature_dim // (2 * input_dim)
        self.omegas = torch.logspace(1, math.log10(sigma_q), self.n)
        self.patch_num = int(math.sqrt(n_patches))
        self.alpha = 10.0
        
        self.omegas_l = [torch.logspace(1, math.log10(sigma_ls[i]), self.n) 
                         for i in range(self.layer_num)]
        
        self.query_lin = nn.Linear(feature_dim, hidden_dim)
        self.modulation_ca = SharedTokenCrossAttention(query_dim=hidden_dim, 
                                                       context_dim=context_dim, heads=2)
        
        self.bandwidth_lins = nn.ModuleList([
            nn.Linear(feature_dim, hidden_dim) for _ in range(self.layer_num)
        ])
        
        self.modulation_lins = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(self.layer_num)
        ])
        
        self.hv_lins = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(len(sigma_ls) - 1)
        ])
        
        self.out_lins = nn.ModuleList([
            nn.Linear(hidden_dim, output_dim) for _ in range(len(sigma_ls))
        ])
        
        self.act = nn.ReLU()
    
    def calc_gamma(self, x, omegas):
        L = x.shape[0]
        coords = x.unsqueeze(-1)
        omegas = omegas.view(1, 1, -1).to(x.device)
        
        arg = torch.pi * coords * omegas
        sin_part = torch.sin(arg)
        cos_part = torch.cos(arg)
        
        gamma = torch.cat([sin_part, cos_part], dim=-1).view(L, -1)
        return gamma
    
    def get_patch_index(self, grid, H, W):
        y = grid[:, 0]
        x = grid[:, 1]
        row = (y * H).to(torch.int32).clamp(0, H-1)
        col = (x * W).to(torch.int32).clamp(0, W-1)
        return row * W + col
    
    def approximate_relative_distances(self, target_index, H, W, m):
        alpha = self.alpha
        N = H * W
        
        t = target_index.float() / N
        token_positions = torch.tensor(
            [(i + 0.5) / m for i in range(m)],
            device=target_index.device
        )
        
        t_expanded = t.unsqueeze(0)
        tokens_expanded = token_positions.unsqueeze(1)
        
        rel_distances = -alpha * torch.abs(t_expanded - tokens_expanded)**2
        
        return rel_distances
    
    def forward(self, x, tokens):
        B, query_shape = x.shape[0], x.shape[1:-1]
        x = x.view(B, -1, x.shape[-1])
        
        grid = x[0]
        indexes = self.get_patch_index(grid, self.patch_num, self.patch_num)
        
        rel_distances = self.approximate_relative_distances(
            indexes, self.patch_num, self.patch_num, tokens.shape[1]
        )
        bias = einops.repeat(rel_distances, 'l n -> b l n', b=B)
        
        x_q = einops.repeat(
            self.calc_gamma(x[0], self.omegas), 'l d -> b l d', b=B
        )
        x_q = self.act(self.query_lin(x_q))
        
        modulation_vector = self.modulation_ca(x_q, context=tokens, bias=bias)
        
        modulations_l = []
        h_f = []
        
        for k in range(self.layer_num):
            x_l = einops.repeat(
                self.calc_gamma(x[0], self.omegas_l[k]), 'l d -> b l d', b=B
            )
            h_l = self.act(self.bandwidth_lins[k](x_l))
            h_f.append(h_l)
            
            m_l = self.act(h_l + self.modulation_lins[k](modulation_vector))
            modulations_l.append(m_l)
        
        h_v = [modulations_l[0]]
        for i in range(self.layer_num - 1):
            h_vl = self.act(self.hv_lins[i](modulations_l[i+1] + h_v[i]))
            h_v.append(h_vl)
        
        outs = [self.out_lins[i](h_v[i]) for i in range(self.layer_num)]
        out = sum(outs)
        
        out = out.view(B, *query_shape, -1)
        return out


print("✓ LAINR decoder defined")

In [ ]:
# ============================================================================
# Complete MAMBA-GINR Model (adapted for 64×64 images)
# ============================================================================

class MambaGINR_VOC(nn.Module):
    """MAMBA-GINR for 64×64 images"""
    def __init__(
        self,
        img_size=64,
        patch_size=2,
        dim=256,
        num_lp=256,
        mamba_depth=6,
        ff_dim=1024,
        lp_type='equidistant',
        feature_dim=64,
        sigma_q=16,
        sigma_ls=[128, 32],
        hidden_dim=256
    ):
        super().__init__()
        
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2  # 1024 for 64×64
        self.dim = dim
        self.patch_num = img_size // patch_size
        
        self.patch_embed = nn.Linear(patch_size * patch_size * 3, dim)
        self.register_buffer('pos_freq', torch.randn(dim // 2, 2) * 10.0)
        self.pos_proj = nn.Linear(dim, dim)
        
        self.lp_module = ImplicitSequentialBias(
            num_lp=num_lp,
            dim=dim,
            input_len=self.num_patches,
            type=lp_type
        )
        
        self.encoder = MambaEncoder(
            depth=mamba_depth,
            dim=dim,
            ff_dim=ff_dim
        )
        
        self.hyponet = LAINRDecoder(
            feature_dim=feature_dim,
            input_dim=2,
            output_dim=3,
            sigma_q=sigma_q,
            sigma_ls=sigma_ls,
            n_patches=self.num_patches,
            hidden_dim=hidden_dim,
            context_dim=dim
        )
    
    def get_patch_positions(self, B, device):
        h = w = self.patch_num
        y = torch.linspace(0.5/h, 1 - 0.5/h, h, device=device)
        x = torch.linspace(0.5/w, 1 - 0.5/w, w, device=device)
        yy, xx = torch.meshgrid(y, x, indexing='ij')
        positions = torch.stack([yy, xx], dim=-1).reshape(-1, 2)
        return positions.unsqueeze(0).expand(B, -1, -1)
    
    def fourier_pos_encoding(self, positions):
        proj = 2 * np.pi * positions @ self.pos_freq.T
        encoding = torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)
        return self.pos_proj(encoding)
    
    def patchify(self, images):
        B, C, H, W = images.shape
        p = self.patch_size
        patches = images.reshape(B, C, H//p, p, W//p, p)
        patches = patches.permute(0, 2, 4, 1, 3, 5).reshape(B, -1, C*p*p)
        return patches
    
    def encode(self, images):
        B = images.shape[0]
        patches = self.patchify(images)
        tokens = self.patch_embed(patches)
        
        positions = self.get_patch_positions(B, images.device)
        pos_encoding = self.fourier_pos_encoding(positions)
        tokens = tokens + pos_encoding
        
        tokens_with_lp = self.lp_module.add_lp(tokens)
        encoded = self.encoder(tokens_with_lp)
        lp_features = self.lp_module.extract_lp(encoded)
        
        return lp_features
    
    def decode(self, lp_features, coords):
        return self.hyponet(coords, lp_features)
    
    def forward(self, images, coords):
        lp_features = self.encode(images)
        return self.decode(lp_features, coords)


def create_coordinate_grid(H, W, device='cpu'):
    y = torch.linspace(0, 1, H, device=device)
    x = torch.linspace(0, 1, W, device=device)
    yy, xx = torch.meshgrid(y, x, indexing='ij')
    coords = torch.stack([yy, xx], dim=-1)
    return coords


print("✓ Complete MAMBA-GINR model defined for 64×64 images")
print(f"  Number of patches: 1024 (32×32)")
print(f"  Number of LP tokens: 256")

## 4. Train MAMBA-GINR on Image Reconstruction (Pretraining)

In [ ]:
# Initialize model
model = MambaGINR_VOC(
    img_size=64,
    patch_size=2,
    dim=256,
    num_lp=256,
    mamba_depth=6,
    hidden_dim=256
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

# Create coordinate grid for reconstruction
coords = create_coordinate_grid(64, 64, device=device)
coords = coords.unsqueeze(0)  # (1, 64, 64, 2)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"Coordinate grid shape: {coords.shape}")


# Training function
def train_reconstruction(model, train_loader, optimizer, epoch):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}')
    
    for images, _ in pbar:  # Ignore masks for reconstruction
        images = images.to(device)
        B = images.shape[0]
        
        # Expand coords for batch
        batch_coords = coords.expand(B, -1, -1, -1)
        
        # Forward pass
        reconstructed = model(images, batch_coords)
        
        # Reconstruction loss
        target = images.permute(0, 2, 3, 1)  # (B, H, W, C)
        loss = F.mse_loss(reconstructed, target)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(train_loader)


def validate_reconstruction(model, val_loader):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for images, _ in val_loader:
            images = images.to(device)
            B = images.shape[0]
            batch_coords = coords.expand(B, -1, -1, -1)
            
            reconstructed = model(images, batch_coords)
            target = images.permute(0, 2, 3, 1)
            loss = F.mse_loss(reconstructed, target)
            
            total_loss += loss.item()
    
    return total_loss / len(val_loader)


# Train MAMBA-GINR
print("\n=== Training MAMBA-GINR on Image Reconstruction ===\n")

train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(1, 51):
    train_loss = train_reconstruction(model, train_loader, optimizer, epoch)
    val_loss = validate_reconstruction(model, val_loader)
    scheduler.step()
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}, LR = {scheduler.get_last_lr()[0]:.2e}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'mamba_ginr_voc_best.pth')
        print(f"  → Best model saved (val_loss={val_loss:.4f})")

print("\n✓ MAMBA-GINR pretraining complete!")

# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('MAMBA-GINR Reconstruction Training')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Visualize reconstructions
model.eval()
with torch.no_grad():
    sample_images, _ = next(iter(val_loader))
    sample_images = sample_images[:4].to(device)
    B = sample_images.shape[0]
    batch_coords = coords.expand(B, -1, -1, -1)
    
    reconstructed = model(sample_images, batch_coords)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i in range(4):
        axes[0, i].imshow(sample_images[i].cpu().permute(1, 2, 0))
        axes[0, i].set_title('Original')
        axes[0, i].axis('off')
        
        recon = reconstructed[i].cpu().clamp(0, 1)
        axes[1, i].imshow(recon)
        axes[1, i].set_title('Reconstructed')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()


## 5. Extract Per-Pixel Modulation Features

In [ ]:
def extract_modulation_features(model, images, resolution=64):
    """
    Extract per-pixel modulation features from LAINR decoder
    
    Args:
        model: Trained MAMBA-GINR model
        images: (B, 3, H, W) input images
        resolution: Image resolution (H=W)
    
    Returns:
        modulation: (B, H, W, hidden_dim) per-pixel modulation features
    """
    model.eval()
    with torch.no_grad():
        B = images.shape[0]
        
        # Step 1: Encode images to get LP features
        lp_features = model.encode(images)  # (B, num_lp, dim)
        
        # Step 2: Create coordinate grid
        coords_grid = create_coordinate_grid(resolution, resolution, device=images.device)
        coords_batch = coords_grid.unsqueeze(0).expand(B, -1, -1, -1)
        
        # Step 3: Extract modulation from LAINR decoder
        # We need to replicate the decoder's query processing
        coords_flat = coords_batch.view(B, -1, 2)  # (B, HW, 2)
        
        # Compute patch indices for spatial bias
        grid = coords_flat[0]
        patch_num = model.hyponet.patch_num
        indexes = model.hyponet.get_patch_index(grid, patch_num, patch_num)
        
        # Compute spatial bias
        rel_distances = model.hyponet.approximate_relative_distances(
            indexes, patch_num, patch_num, lp_features.shape[1]
        )
        bias = einops.repeat(rel_distances, 'l n -> b l n', b=B)
        
        # Fourier encoding of queries
        x_q = einops.repeat(
            model.hyponet.calc_gamma(coords_flat[0], model.hyponet.omegas),
            'l d -> b l d',
            b=B
        )
        x_q = F.relu(model.hyponet.query_lin(x_q))
        
        # Extract modulation via cross-attention
        modulation_vector = model.hyponet.modulation_ca(x_q, context=lp_features, bias=bias)
        # modulation_vector: (B, HW, hidden_dim)
        
        # Reshape to spatial dimensions
        modulation = modulation_vector.view(B, resolution, resolution, -1)
        
    return modulation


# Load best model
model.load_state_dict(torch.load('mamba_ginr_voc_best.pth'))
print("✓ Loaded best MAMBA-GINR model")

# Extract modulation features for entire dataset
print("\n=== Extracting Modulation Features ===\n")

def extract_all_features(model, dataloader, desc="Extracting"):
    """Extract modulation features for entire dataset"""
    all_modulations = []
    all_masks = []
    all_images = []
    
    for images, masks in tqdm(dataloader, desc=desc):
        images = images.to(device)
        
        # Extract modulation features
        modulation = extract_modulation_features(model, images, resolution=64)
        
        all_modulations.append(modulation.cpu())
        all_masks.append(masks)
        all_images.append(images.cpu())
    
    modulations = torch.cat(all_modulations, dim=0)  # (N, 64, 64, 256)
    masks = torch.cat(all_masks, dim=0)  # (N, 64, 64)
    images = torch.cat(all_images, dim=0)  # (N, 3, 64, 64)
    
    return modulations, masks, images


# Extract features
train_modulations, train_masks, train_images = extract_all_features(model, train_loader, "Train")
val_modulations, val_masks, val_images = extract_all_features(model, val_loader, "Val")

print(f"\nTrain modulation features: {train_modulations.shape}")
print(f"Train masks: {train_masks.shape}")
print(f"Train images: {train_images.shape}")
print(f"\nVal modulation features: {val_modulations.shape}")
print(f"Val masks: {val_masks.shape}")
print(f"Val images: {val_images.shape}")

# Visualize modulation features
print("\n=== Visualizing Modulation Features ===")

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(4):
    # Original image
    axes[0, i].imshow(train_images[i].permute(1, 2, 0))
    axes[0, i].set_title('Original Image')
    axes[0, i].axis('off')
    
    # Modulation feature (first 3 channels as RGB)
    mod_rgb = train_modulations[i, :, :, :3].numpy()
    mod_rgb = (mod_rgb - mod_rgb.min()) / (mod_rgb.max() - mod_rgb.min() + 1e-8)
    axes[1, i].imshow(mod_rgb)
    axes[1, i].set_title('Modulation (first 3 dims)')
    axes[1, i].axis('off')
    
    # Segmentation mask
    axes[2, i].imshow(mask_to_rgb(train_masks[i].numpy()))
    axes[2, i].set_title('Ground Truth Mask')
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()

print("\n✓ Modulation feature extraction complete!")


## 6. U-Net Segmentation Architecture

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive 3x3 convolutions"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    """
    U-Net for semantic segmentation
    
    Args:
        in_channels: Input channels (3 for RGB, 256 for modulation)
        num_classes: Number of output classes (21 for VOC)
    """
    def __init__(self, in_channels=3, num_classes=21):
        super().__init__()
        
        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        
        self.enc4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)
        
        # Decoder
        self.upconv4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        
        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        
        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        
        # Output
        self.out = nn.Conv2d(64, num_classes, 1)
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        
        e3 = self.enc3(p2)
        p3 = self.pool3(e3)
        
        e4 = self.enc4(p3)
        p4 = self.pool4(e4)
        
        # Bottleneck
        b = self.bottleneck(p4)
        
        # Decoder with skip connections
        d4 = self.upconv4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)
        
        d3 = self.upconv3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)
        
        d2 = self.upconv2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        
        d1 = self.upconv1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        
        # Output
        out = self.out(d1)
        return out


# Create models
unet_baseline = UNet(in_channels=3, num_classes=21).to(device)  # RGB baseline
unet_proposed = UNet(in_channels=256, num_classes=21).to(device)  # Modulation features

print(f"U-Net Baseline (RGB) parameters: {sum(p.numel() for p in unet_baseline.parameters()) / 1e6:.2f}M")
print(f"U-Net Proposed (Modulation) parameters: {sum(p.numel() for p in unet_proposed.parameters()) / 1e6:.2f}M")
print("\n✓ U-Net architectures defined")


## 7. Prepare Segmentation Datasets

In [ ]:
class SegmentationDataset(Dataset):
    """Dataset for segmentation with either RGB or modulation features"""
    def __init__(self, features, masks, feature_type='rgb'):
        """
        Args:
            features: (N, H, W, C) for modulation or (N, C, H, W) for RGB
            masks: (N, H, W) segmentation masks
            feature_type: 'rgb' or 'modulation'
        """
        self.features = features
        self.masks = masks
        self.feature_type = feature_type
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        mask = self.masks[idx]  # (H, W)
        
        if self.feature_type == 'rgb':
            # RGB: already (C, H, W)
            feature = self.features[idx]
        else:
            # Modulation: (H, W, C) → (C, H, W)
            feature = self.features[idx].permute(2, 0, 1)
        
        return feature, mask


# Create datasets
train_dataset_baseline = SegmentationDataset(train_images, train_masks, feature_type='rgb')
val_dataset_baseline = SegmentationDataset(val_images, val_masks, feature_type='rgb')

train_dataset_proposed = SegmentationDataset(train_modulations, train_masks, feature_type='modulation')
val_dataset_proposed = SegmentationDataset(val_modulations, val_masks, feature_type='modulation')

# Create dataloaders
train_loader_baseline = DataLoader(train_dataset_baseline, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader_baseline = DataLoader(val_dataset_baseline, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

train_loader_proposed = DataLoader(train_dataset_proposed, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader_proposed = DataLoader(val_dataset_proposed, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

print("Baseline (RGB) datasets:")
print(f"  Train: {len(train_dataset_baseline)} samples")
print(f"  Val: {len(val_dataset_baseline)} samples")
print(f"\nProposed (Modulation) datasets:")
print(f"  Train: {len(train_dataset_proposed)} samples")
print(f"  Val: {len(val_dataset_proposed)} samples")
print("\n✓ Segmentation datasets prepared")


## 8. Segmentation Training and Evaluation

In [ ]:
def compute_iou(pred, target, num_classes=21, ignore_index=21):
    """
    Compute Intersection over Union (IoU) for each class
    
    Args:
        pred: (N, H, W) predicted class labels
        target: (N, H, W) ground truth labels
        num_classes: Number of classes (excluding ignore class)
        ignore_index: Class to ignore (21 for VOC)
    
    Returns:
        iou_per_class: (num_classes,) IoU for each class
        mean_iou: Mean IoU across all classes
    """
    ious = []
    
    for cls in range(num_classes):
        pred_mask = (pred == cls)
        target_mask = (target == cls)
        
        # Intersection and union
        intersection = (pred_mask & target_mask).sum().float()
        union = (pred_mask | target_mask).sum().float()
        
        if union == 0:
            # Class not present in batch
            ious.append(float('nan'))
        else:
            iou = intersection / union
            ious.append(iou.item())
    
    # Filter out NaN values for mean calculation
    valid_ious = [iou for iou in ious if not np.isnan(iou)]
    mean_iou = np.mean(valid_ious) if valid_ious else 0.0
    
    return np.array(ious), mean_iou


def train_segmentation(model, train_loader, optimizer, epoch):
    """Train segmentation model for one epoch"""
    model.train()
    total_loss = 0
    total_iou = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}')
    
    for features, masks in pbar:
        features = features.to(device)
        masks = masks.to(device)
        
        # Forward pass
        logits = model(features)
        
        # Compute loss (ignore class 21)
        loss = F.cross_entropy(logits, masks, ignore_index=21)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        # Compute IoU
        pred = logits.argmax(dim=1)
        _, iou = compute_iou(pred.cpu(), masks.cpu())
        
        total_loss += loss.item()
        total_iou += iou
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'iou': f'{iou:.4f}'})
    
    avg_loss = total_loss / len(train_loader)
    avg_iou = total_iou / len(train_loader)
    
    return avg_loss, avg_iou


def evaluate_segmentation(model, val_loader):
    """Evaluate segmentation model"""
    model.eval()
    total_loss = 0
    total_iou = 0
    all_class_ious = []
    
    with torch.no_grad():
        for features, masks in val_loader:
            features = features.to(device)
            masks = masks.to(device)
            
            # Forward pass
            logits = model(features)
            
            # Compute loss
            loss = F.cross_entropy(logits, masks, ignore_index=21)
            
            # Compute IoU
            pred = logits.argmax(dim=1)
            class_ious, mean_iou = compute_iou(pred.cpu(), masks.cpu())
            
            total_loss += loss.item()
            total_iou += mean_iou
            all_class_ious.append(class_ious)
    
    avg_loss = total_loss / len(val_loader)
    avg_iou = total_iou / len(val_loader)
    
    # Average class IoUs across batches
    class_ious_array = np.array(all_class_ious)
    avg_class_ious = np.nanmean(class_ious_array, axis=0)
    
    return avg_loss, avg_iou, avg_class_ious


print("✓ Training and evaluation functions defined")


## 9. Train Baseline U-Net (RGB Input)

In [ ]:
print("\n" + "="*60)
print("BASELINE: U-Net on Raw RGB (64×64×3)")
print("="*60 + "\n")

# Initialize optimizer and scheduler
optimizer_baseline = torch.optim.AdamW(unet_baseline.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_baseline = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_baseline, T_max=30, eta_min=1e-6)

# Training loop
train_losses_baseline = []
val_losses_baseline = []
train_ious_baseline = []
val_ious_baseline = []
best_iou_baseline = 0.0

for epoch in range(1, 31):
    train_loss, train_iou = train_segmentation(unet_baseline, train_loader_baseline, optimizer_baseline, epoch)
    val_loss, val_iou, val_class_ious = evaluate_segmentation(unet_baseline, val_loader_baseline)
    scheduler_baseline.step()
    
    train_losses_baseline.append(train_loss)
    val_losses_baseline.append(val_loss)
    train_ious_baseline.append(train_iou)
    val_ious_baseline.append(val_iou)
    
    print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Train IoU={train_iou:.4f} | "
          f"Val Loss={val_loss:.4f}, Val IoU={val_iou:.4f}")
    
    # Save best model
    if val_iou > best_iou_baseline:
        best_iou_baseline = val_iou
        torch.save(unet_baseline.state_dict(), 'unet_baseline_best.pth')
        print(f"  → Best baseline model saved (val_iou={val_iou:.4f})")

print(f"\n✓ Baseline training complete! Best Val IoU: {best_iou_baseline:.4f}")


## 10. Train Proposed U-Net (Modulation Features)

In [ ]:
print("\n" + "="*60)
print("PROPOSED: U-Net on Modulation Features (64×64×256)")
print("="*60 + "\n")

# Initialize optimizer and scheduler
optimizer_proposed = torch.optim.AdamW(unet_proposed.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_proposed = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_proposed, T_max=30, eta_min=1e-6)

# Training loop
train_losses_proposed = []
val_losses_proposed = []
train_ious_proposed = []
val_ious_proposed = []
best_iou_proposed = 0.0

for epoch in range(1, 31):
    train_loss, train_iou = train_segmentation(unet_proposed, train_loader_proposed, optimizer_proposed, epoch)
    val_loss, val_iou, val_class_ious = evaluate_segmentation(unet_proposed, val_loader_proposed)
    scheduler_proposed.step()
    
    train_losses_proposed.append(train_loss)
    val_losses_proposed.append(val_loss)
    train_ious_proposed.append(train_iou)
    val_ious_proposed.append(val_iou)
    
    print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Train IoU={train_iou:.4f} | "
          f"Val Loss={val_loss:.4f}, Val IoU={val_iou:.4f}")
    
    # Save best model
    if val_iou > best_iou_proposed:
        best_iou_proposed = val_iou
        torch.save(unet_proposed.state_dict(), 'unet_proposed_best.pth')
        print(f"  → Best proposed model saved (val_iou={val_iou:.4f})")

print(f"\n✓ Proposed training complete! Best Val IoU: {best_iou_proposed:.4f}")


## 11. Results Comparison and Analysis

In [ ]:
print("\n" + "="*80)
print("FINAL COMPARISON: Baseline (RGB) vs Proposed (Modulation Features)")
print("="*80 + "\n")

# Load best models
unet_baseline.load_state_dict(torch.load('unet_baseline_best.pth'))
unet_proposed.load_state_dict(torch.load('unet_proposed_best.pth'))

# Final evaluation
_, baseline_iou, baseline_class_ious = evaluate_segmentation(unet_baseline, val_loader_baseline)
_, proposed_iou, proposed_class_ious = evaluate_segmentation(unet_proposed, val_loader_proposed)

print(f"Baseline (RGB) Mean IoU: {baseline_iou:.4f}")
print(f"Proposed (Modulation) Mean IoU: {proposed_iou:.4f}")
print(f"Improvement: {(proposed_iou - baseline_iou):.4f} ({(proposed_iou - baseline_iou) / baseline_iou * 100:.2f}%)")

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss curves
axes[0].plot(train_losses_baseline, label='Baseline Train', color='blue', linestyle='--')
axes[0].plot(val_losses_baseline, label='Baseline Val', color='blue')
axes[0].plot(train_losses_proposed, label='Proposed Train', color='red', linestyle='--')
axes[0].plot(val_losses_proposed, label='Proposed Val', color='red')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# IoU curves
axes[1].plot(train_ious_baseline, label='Baseline Train', color='blue', linestyle='--')
axes[1].plot(val_ious_baseline, label='Baseline Val', color='blue')
axes[1].plot(train_ious_proposed, label='Proposed Train', color='red', linestyle='--')
axes[1].plot(val_ious_proposed, label='Proposed Val', color='red')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean IoU')
axes[1].set_title('Mean IoU Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Per-class IoU comparison
print("\n" + "="*80)
print("Per-Class IoU Comparison")
print("="*80 + "\n")

class_names = VOC_CLASSES[:21]
comparison_data = []

for i, class_name in enumerate(class_names):
    baseline_val = baseline_class_ious[i]
    proposed_val = proposed_class_ious[i]
    
    if not np.isnan(baseline_val) and not np.isnan(proposed_val):
        improvement = proposed_val - baseline_val
        comparison_data.append({
            'Class': class_name,
            'Baseline': baseline_val,
            'Proposed': proposed_val,
            'Improvement': improvement
        })

# Sort by improvement
comparison_data.sort(key=lambda x: x['Improvement'], reverse=True)

print(f"{'Class':<20} {'Baseline IoU':<15} {'Proposed IoU':<15} {'Improvement':<15}")
print("-" * 70)
for row in comparison_data:
    print(f"{row['Class']:<20} {row['Baseline']:<15.4f} {row['Proposed']:<15.4f} {row['Improvement']:<15.4f}")

# Bar chart comparison
fig, ax = plt.subplots(figsize=(16, 6))
classes = [row['Class'] for row in comparison_data]
baseline_vals = [row['Baseline'] for row in comparison_data]
proposed_vals = [row['Proposed'] for row in comparison_data]

x = np.arange(len(classes))
width = 0.35

ax.bar(x - width/2, baseline_vals, width, label='Baseline (RGB)', color='blue', alpha=0.7)
ax.bar(x + width/2, proposed_vals, width, label='Proposed (Modulation)', color='red', alpha=0.7)

ax.set_xlabel('Class')
ax.set_ylabel('IoU')
ax.set_title('Per-Class IoU: Baseline vs Proposed')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n✓ Comparison complete!")


## 12. Qualitative Visualization

In [ ]:
print("\n=== Qualitative Visualization ===\n")

# Get sample predictions
unet_baseline.eval()
unet_proposed.eval()

num_samples = 8
sample_indices = np.random.choice(len(val_dataset_baseline), num_samples, replace=False)

fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))

for idx, sample_idx in enumerate(sample_indices):
    # Get data
    rgb_feature, mask_gt = val_dataset_baseline[sample_idx]
    mod_feature, _ = val_dataset_proposed[sample_idx]
    
    # Predictions
    with torch.no_grad():
        rgb_feature_batch = rgb_feature.unsqueeze(0).to(device)
        mod_feature_batch = mod_feature.unsqueeze(0).to(device)
        
        pred_baseline = unet_baseline(rgb_feature_batch).argmax(dim=1).squeeze().cpu()
        pred_proposed = unet_proposed(mod_feature_batch).argmax(dim=1).squeeze().cpu()
    
    # Plot
    axes[idx, 0].imshow(rgb_feature.permute(1, 2, 0))
    axes[idx, 0].set_title('Input Image')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(mask_to_rgb(mask_gt.numpy()))
    axes[idx, 1].set_title('Ground Truth')
    axes[idx, 1].axis('off')
    
    axes[idx, 2].imshow(mask_to_rgb(pred_baseline.numpy()))
    axes[idx, 2].set_title('Baseline Prediction')
    axes[idx, 2].axis('off')
    
    axes[idx, 3].imshow(mask_to_rgb(pred_proposed.numpy()))
    axes[idx, 3].set_title('Proposed Prediction')
    axes[idx, 3].axis('off')

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete!")

# Summary
print("\n" + "="*80)
print("EXPERIMENT SUMMARY")
print("="*80)
print(f"""
Research Question: Do learned modulation features from reconstruction pretraining 
                   improve semantic segmentation?

Experimental Setup:
  1. Pretrained MAMBA-GINR on 64×64 image reconstruction (50 epochs)
  2. Extracted per-pixel modulation features (64×64×256)
  3. Trained two U-Net segmentation models (30 epochs each):
     - Baseline: U-Net on raw RGB (64×64×3)
     - Proposed: U-Net on modulation features (64×64×256)

Results:
  - Baseline (RGB) Mean IoU: {baseline_iou:.4f}
  - Proposed (Modulation) Mean IoU: {proposed_iou:.4f}
  - Improvement: {(proposed_iou - baseline_iou):.4f} ({(proposed_iou - baseline_iou) / baseline_iou * 100:.2f}%)

Conclusion:
  {"✅ SUCCESS! " if proposed_iou > baseline_iou else "❌ FAILED: "}
  {"Modulation features learned through reconstruction DO improve segmentation!" 
   if proposed_iou > baseline_iou 
   else "Modulation features did not outperform raw RGB."}
  
  This {"supports" if proposed_iou > baseline_iou else "challenges"} the hypothesis 
  that reconstruction pretraining learns semantically meaningful features.
""")